This is for exploring Bayesian optimization with Naman.

In [2]:
#run this to see if PMM working

import sys
import os
# Add the parent folder (PMM-Design) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from PMM.PMMInSitu import PMMInSitu
PMM = PMMInSitu('../confs/conf_test.yaml')

# Run these 2 lines:
# PMM.Set_Bulb_VI('all', 10, 2) #set bulbs(all, volts, amps)
# PMM.Activate_Bulb('all')

# OR run these 2 lines:
# # PMM.Config_Check()
PMM.Deactivate_Bulb('all')

In [ ]:
# Finding Epsilon

#MAKE SURE TO EMPTY OUTPUT FOLDER FIRST

# This is for collecting preliminary data
# epsilon calibration
# S21 with obj_dB at 6 GHz
# Resume-capable + per-direction checkpointing
#
# CHANGES (minimal):
#  1) Recompute center objective every direction (drift-safe)
#  2) Clip starting points into an interior box so rho0 +/- eps has room (reduced boundary clipping)
#  3) FIX: resume summary checkpoint read handles empty file (prevents EmptyDataError)
#  4) FIX: retry
# With averaged center

import os
import sys
import numpy as np
import pandas as pd
from datetime import datetime
import time  

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from PMM.PMMInSitu import PMMInSitu

# ----------------------------
# User settings
# ----------------------------
conf_file = '../confs/conf_test.yaml'
conf_dir = '../confs/'

save_dir = '../outputs/bayes_exploration'
os.makedirs(save_dir, exist_ok=True)

# Objective settings
f_target_GHz = 6.0
df_GHz = 0.5  # can change if you want narrower/wider dB integration band
objective_name = 'comp' #dB
duty_cycle = 0.5

# Hardware mapping settings (same style as your Bayes runs)
fpm = 14.65  # GHz max plasma frequency parameter
k = 0.33
S = 1.0

# Epsilon candidates specified in *GHz-equivalent* scale, then converted to rho-space
eps_GHz_list = np.array([0.3, 0.1, 0.03], dtype=float)

# Direction sampling
n_dirs = 8
rng_seed = 20260224
rng = np.random.default_rng(rng_seed)  # fixed seed for reproducibility

# Resume settings
# - "YYYYMMDD_HHMMSS" -> resume from that run's checkpoint files
resume_run_stamp = "20260305_010100"  # None        #UPDATE THIS

# Optionally limit starts for testing/resume:
starts_to_run = [
    "optimized",
    # "intuitive",
    # "random",
    # "cold"        #UPDATE THIS
]

# If True, and resuming, skip directions already saved in checkpoint all CSV
resume_skip_completed = True

# If True, skip a start entirely if summary checkpoint already has that start
skip_start_if_summary_exists = False  # set True if you want clean "finished start" skipping by summary

# ----------------------------
# Initialize PMM
# ----------------------------
PMM = PMMInSitu(conf_file, conf_dir=conf_dir)

num_bulbs = sum(len(v) for v in PMM.config["serial_ports"].values())
rho_upper = PMM.f_a(fpm)  # internal rho-space upper bound corresponding to fpm [GHz]
print(f"num_bulbs = {num_bulbs}")
print(f"rho_upper = PMM.f_a({fpm} GHz) = {rho_upper:.6f} (a-units)")

# Convert epsilon list from GHz to rho-space
eps_rho_list = np.array([PMM.f_a(eps) for eps in eps_GHz_list], dtype=float)
print("Epsilon candidates:")
for eG, er in zip(eps_GHz_list, eps_rho_list):
    print(f"  {eG:.3f} GHz  ->  {er:.6e} rho-units")

# --- NEW (minimal): interior bounds based on largest epsilon ---
eps_rho_max = float(np.max(eps_rho_list))
rho_lo_interior = eps_rho_max
rho_hi_interior = rho_upper - eps_rho_max
print(f"Interior bounds: [{rho_lo_interior:.6e}, {rho_hi_interior:.6e}]")

# ----------------------------
# Helper functions
# ----------------------------
def unit_random_directions(n, dim, rng):
    D = rng.standard_normal((n, dim))
    norms = np.linalg.norm(D, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return D / norms

def clip_rho(rho, lo=0.0, hi=None):
    if hi is None:
        return np.clip(rho, lo, None)
    return np.clip(rho, lo, hi)

# --- NEW (minimal): clip starts into interior so +/- eps has room ---
def make_interior(rho, lo, hi):
    return np.clip(rho, lo, hi)

def evaluate_wvg_db(PMM, rho, fpm, k, S, f_GHz, df_GHz, norms, duty_cycle=0.5):
    """
    One objective evaluation using PMM.Wvg_Obj_Get.
    If norms is None, first call computes normalization constants.
    Shift by +1 so the starting state is roughly 1 instead of 0.
    """
    if norms is None:
        _, norms_out = PMM.Wvg_Obj_Get(
            rho, fpm, k, S, f_GHz, df_GHz,
            objective=objective_name, norms=[], duty_cycle=duty_cycle
        )
        obj, _ = PMM.Wvg_Obj_Get(
            rho, fpm, k, S, f_GHz, df_GHz,
            objective=objective_name, norms=norms_out, duty_cycle=duty_cycle
        )
        return obj + 1.0, norms_out
    else:
        obj, _ = PMM.Wvg_Obj_Get(
            rho, fpm, k, S, f_GHz, df_GHz,
            objective=objective_name, norms=norms, duty_cycle=duty_cycle
        )
        return obj + 1.0, norms

# --- NEW (minimal): retry wrapper for the specific hardware error ---
def evaluate_wvg_db_retry(PMM, rho, fpm, k, S, f_GHz, df_GHz, norms, duty_cycle=0.5,
                          max_tries=10, sleep_s=1.0):
    """
    Same as evaluate_wvg_db, but retries max_tries times if we hit:
    "device not configured"
    """
    last_err = None
    for attempt in range(1, max_tries + 1):
        try:
            return evaluate_wvg_db(PMM, rho, fpm, k, S, f_GHz, df_GHz, norms, duty_cycle=duty_cycle)
        except Exception as e:
            last_err = e
            msg = str(e).lower()
            if "device not configured" in msg:
                print(f"[retry] device not configured (attempt {attempt}/{max_tries})")
                if attempt < max_tries:
                    time.sleep(sleep_s)
                    continue
                raise RuntimeError(f"device not configured after {max_tries} tries") from e
            else:
                raise
    raise last_err

def central_fwd_bwd_disagreement(f_plus, f_minus, f_center_avg, eps):
    """
    Returns central, forward, backward finite differences and disagreement.
    disagreement = |fwd - bwd| / |central| (NaN if |central| ~ 0)

    Here f_center_avg should be the average of the two center measurements.
    """
    central = (f_plus - f_minus) / (2.0 * eps)
    fwd = (f_plus - f_center_avg) / eps
    bwd = (f_center_avg - f_minus) / eps
    if abs(central) > 1e-12:
        disagreement = abs(fwd - bwd) / abs(central)
    else:
        disagreement = np.nan
    return central, fwd, bwd, disagreement

def save_checkpoint(save_dir, run_stamp, all_rows, summary_rows, D, tag="checkpoint"):
    """
    Save partial progress so hardware errors don't lose the whole run.
    """
    df_all_ckpt = pd.DataFrame(all_rows)
    df_summary_ckpt = pd.DataFrame(summary_rows)

    all_ckpt_csv = os.path.join(save_dir, f"epsilon_calibration_all_{run_stamp}_{tag}.csv")
    summary_ckpt_csv = os.path.join(save_dir, f"epsilon_calibration_summary_{run_stamp}_{tag}.csv")
    dirs_ckpt_csv = os.path.join(save_dir, f"epsilon_calibration_directions_{run_stamp}_{tag}.csv")

    df_all_ckpt.to_csv(all_ckpt_csv, index=False)
    df_summary_ckpt.to_csv(summary_ckpt_csv, index=False)
    pd.DataFrame(D).to_csv(dirs_ckpt_csv, index=False)

    return all_ckpt_csv, summary_ckpt_csv, dirs_ckpt_csv

def dedupe_summary_rows(summary_rows):
    """
    Keep only the latest summary entry for each (start_name, eps_GHz).
    Helpful when resuming and re-running a partially completed start.
    """
    if len(summary_rows) == 0:
        return summary_rows
    df = pd.DataFrame(summary_rows)
    if df.empty:
        return summary_rows
    df = df.drop_duplicates(subset=["start_name", "eps_GHz"], keep="last")
    return df.to_dict(orient="records")

# ----------------------------
# Define the 4 starts
# ----------------------------
# 1) cold start (all zeros)
rho_cold = np.zeros(num_bulbs, dtype=float)

# 2) intuitive start:
# Turn only selected bulbs "on" at 0.75*fpm (in rho-space), leave all others off.
rho_intuitive = np.zeros(num_bulbs, dtype=float)
bulbs_to_turn_on = [
    41, 52, 62, 71, 79, 86, 1, 2, 3, 4, 5, 6, 9, 10, 11, 12, 13,
    18, 19, 20, 21, 27, 28, 29, 30, 37, 38, 39, 40, 48, 49, 50, 51,
    59, 60, 61, 68, 69, 70, 77, 78, 84, 85, 91
]

fp_intuitive_on = 0.75 * fpm  # GHz
rho_intuitive_on = PMM.f_a(fp_intuitive_on)  # rho-units

idx_on = np.array(bulbs_to_turn_on, dtype=int) - 1  # bulb addresses are 1-indexed
if np.any(idx_on < 0) or np.any(idx_on >= num_bulbs):
    bad = [b for b in bulbs_to_turn_on if (b < 1 or b > num_bulbs)]
    raise ValueError(f"Invalid bulb addresses in bulbs_to_turn_on: {bad}")

rho_intuitive[idx_on] = rho_intuitive_on
print(f"Intuitive start: {len(idx_on)} bulbs ON at fp={fp_intuitive_on:.3f} GHz "
      f"(rho={rho_intuitive_on:.6e}), others OFF")

# 3) optimized start (manual CSV + choose row explicitly)
optimized_csv_name = 'rho_best_manual_6GHz.csv'  # <-- change to your file if needed
optimized_csv_path = os.path.join(save_dir, optimized_csv_name)

optimized_row = 27  # <-- choose row in CSV

if not os.path.isfile(optimized_csv_path):
    raise FileNotFoundError(
        f"Could not find optimized start CSV: {optimized_csv_path}\n"
        "Put the file in save_dir and update optimized_csv_name."
    )

rho_csv = np.loadtxt(optimized_csv_path, delimiter=',')
rho_csv = np.array(rho_csv, dtype=float)

if rho_csv.ndim == 1:
    rho_optimized = rho_csv.copy()
    print("[info] optimized CSV is 1D (single row). Using that row.")
elif rho_csv.ndim == 2:
    n_rows, n_cols = rho_csv.shape
    if not (-n_rows <= optimized_row < n_rows):
        raise IndexError(f"optimized_row={optimized_row} out of range for CSV with {n_rows} rows")
    rho_optimized = rho_csv[optimized_row, :].copy()
    print(f"[info] optimized CSV shape={rho_csv.shape}; using row {optimized_row}.")
else:
    raise ValueError(f"Unexpected optimized CSV shape: {rho_csv.shape}")

if rho_optimized.size != num_bulbs:
    raise ValueError(
        f"optimized rho length {rho_optimized.size} does not match num_bulbs {num_bulbs}"
    )

rho_optimized = np.clip(rho_optimized, 0.0, rho_upper)

# 4) random start
rho_random = rng.uniform(low=0.0, high=rho_upper, size=num_bulbs)

# Clip all starts just in case
rho_cold = clip_rho(rho_cold, 0.0, rho_upper)
rho_intuitive = clip_rho(rho_intuitive, 0.0, rho_upper)
rho_optimized = clip_rho(rho_optimized, 0.0, rho_upper)
rho_random = clip_rho(rho_random, 0.0, rho_upper)

# --- NEW (minimal): push starts into interior box so +/- eps has room ---
rho_cold = make_interior(rho_cold, rho_lo_interior, rho_hi_interior)
rho_intuitive = make_interior(rho_intuitive, rho_lo_interior, rho_hi_interior)
rho_optimized = make_interior(rho_optimized, rho_lo_interior, rho_hi_interior)
rho_random = make_interior(rho_random, rho_lo_interior, rho_hi_interior)

starts_all = {
    "optimized": rho_optimized,
    "intuitive": rho_intuitive,
    "cold": rho_cold,
    "random": rho_random,
}

# Respect starts_to_run order and names
starts = {}
for nm in starts_to_run:
    if nm not in starts_all:
        raise ValueError(f"Unknown start name in starts_to_run: {nm}")
    starts[nm] = starts_all[nm]

# ----------------------------
# Main epsilon-finding loop (resume-aware)
# ----------------------------
all_rows = []
summary_rows = []

# same 8 directions used across starts (critical for reproducibility + resume)
# IMPORTANT: must use the same seed and same generation call path before this line
D = unit_random_directions(n_dirs, num_bulbs, rng)

# Run stamp handling
if resume_run_stamp is None:
    run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
else:
    run_stamp = str(resume_run_stamp)

print(f"\nRun stamp: {run_stamp}")

# Resume state
completed_keys = set()       # keys of finished rows: (start_name, eps_GHz, dir_idx)
summary_done_starts = set()  # optional skip-whole-start behavior if summary exists

ckpt_all_path = os.path.join(save_dir, f"epsilon_calibration_all_{run_stamp}_checkpoint.csv")
ckpt_summary_path = os.path.join(save_dir, f"epsilon_calibration_summary_{run_stamp}_checkpoint.csv")
ckpt_dirs_path = os.path.join(save_dir, f"epsilon_calibration_directions_{run_stamp}_checkpoint.csv")

if resume_run_stamp is not None:
    if os.path.isfile(ckpt_all_path):
        df_prev_all = pd.read_csv(ckpt_all_path)
        print(f"[resume] Loaded checkpoint raw rows: {len(df_prev_all)} from")
        print("  ", ckpt_all_path)

        # Rebuild in-memory all_rows so future checkpoints/final saves include prior progress
        all_rows = df_prev_all.to_dict(orient="records")

        # Mark completed direction rows
        for _, rr in df_prev_all.iterrows():
            try:
                key = (str(rr["start_name"]), float(rr["eps_GHz"]), int(rr["dir_idx"]))
                completed_keys.add(key)
            except Exception:
                pass
    else:
        print(f"[resume] No raw checkpoint found at {ckpt_all_path}. Starting from scratch under this run_stamp.")

    # ---- FIX (minimal): avoid EmptyDataError when summary checkpoint exists but is empty/blank ----
    if os.path.isfile(ckpt_summary_path) and os.path.getsize(ckpt_summary_path) > 0:
        try:
            df_prev_summary = pd.read_csv(ckpt_summary_path)
            print(f"[resume] Loaded checkpoint summary rows: {len(df_prev_summary)} from")
            print("  ", ckpt_summary_path)

            summary_rows = df_prev_summary.to_dict(orient="records")
            if not df_prev_summary.empty and "start_name" in df_prev_summary.columns:
                summary_done_starts = set(df_prev_summary["start_name"].astype(str).unique().tolist())
        except pd.errors.EmptyDataError:
            print(f"[resume] Summary checkpoint is non-empty bytes but has no columns: {ckpt_summary_path}")
    else:
        print(f"[resume] No usable summary checkpoint at {ckpt_summary_path} (missing or empty).")

    # Optional warning about directions file
    if not os.path.isfile(ckpt_dirs_path):
        print(f"[resume] Warning: directions checkpoint not found at {ckpt_dirs_path}. "
              "We will regenerate D from the fixed RNG seed.")

# Main loop
for start_name, rho0 in starts.items():
    if skip_start_if_summary_exists and (start_name in summary_done_starts):
        print("\n" + "="*90)
        print(f"START: {start_name} [SKIP summary already exists in checkpoint]")
        print("="*90)
        continue

    print("\n" + "="*90)
    print(f"START: {start_name}")
    print("="*90)

    # Compute norms + a center objective at beginning (kept, but we will ALSO re-center per direction)
    f_center_begin, norms = evaluate_wvg_db_retry(
        PMM, rho0, fpm, k, S, f_target_GHz, df_GHz, norms=None, duty_cycle=duty_cycle
    )
    print(f"[{start_name}] center(begin) = {f_center_begin:.6f}")

    # Loop over epsilon candidates and directions
    for eps_GHz, eps_rho in zip(eps_GHz_list, eps_rho_list):
        print(f"\n  Epsilon = {eps_GHz:.3f} GHz-equivalent  (rho eps = {eps_rho:.6e})")

        for d_idx in range(n_dirs):
            d = D[d_idx]

            # Resume skip: if this direction row already exists, skip it
            resume_key = (start_name, float(eps_GHz), int(d_idx))
            if resume_skip_completed and (resume_key in completed_keys):
                print(f"    dir {d_idx+1:02d}/{n_dirs}: [SKIP already saved]")
                continue

            try:
                rho_plus = clip_rho(rho0 + eps_rho * d, 0.0, rho_upper)
                rho_minus = clip_rho(rho0 - eps_rho * d, 0.0, rho_upper)

                # Track clipping severity
                raw_plus = rho0 + eps_rho * d
                raw_minus = rho0 - eps_rho * d
                n_clip_plus = int(np.sum((raw_plus < 0.0) | (raw_plus > rho_upper)))
                n_clip_minus = int(np.sum((raw_minus < 0.0) | (raw_minus > rho_upper)))

                # --- NEW (minimal): center measured before and after +/- ---
                f_center_1, _ = evaluate_wvg_db_retry(
                    PMM, rho0, fpm, k, S, f_target_GHz, df_GHz,
                    norms=norms, duty_cycle=duty_cycle
                )

                # +/- passes
                f_plus, _ = evaluate_wvg_db_retry(
                    PMM, rho_plus, fpm, k, S, f_target_GHz, df_GHz,
                    norms=norms, duty_cycle=duty_cycle
                )
                f_minus, _ = evaluate_wvg_db_retry(
                    PMM, rho_minus, fpm, k, S, f_target_GHz, df_GHz,
                    norms=norms, duty_cycle=duty_cycle
                )

                f_center_2, _ = evaluate_wvg_db_retry(
                    PMM, rho0, fpm, k, S, f_target_GHz, df_GHz,
                    norms=norms, duty_cycle=duty_cycle
                )

                f_center = 0.5 * (f_center_1 + f_center_2)

                central, fwd, bwd, disagreement = central_fwd_bwd_disagreement(
                    f_plus=f_plus, f_minus=f_minus, f_center_avg=f_center, eps=eps_rho
                )

                row = {
                    "run_stamp": run_stamp,
                    "start_name": start_name,
                    "objective": objective_name,
                    "f_target_GHz": f_target_GHz,
                    "df_GHz": df_GHz,
                    "fpm_GHz": fpm,
                    "k": k,
                    "S": S,

                    "dir_idx": d_idx,
                    "eps_GHz": float(eps_GHz),
                    "eps_rho": float(eps_rho),

                    # NOTE: these are objective values, not frequencies
                    # Keep column name "f_center_begin" to avoid breaking downstream scripts,
                    # but it now stores the averaged per-direction center value.
                    "f_center_begin": float(f_center),
                    "f_center_1": float(f_center_1),
                    "f_center_2": float(f_center_2),
                    "f_plus": float(f_plus),
                    "f_minus": float(f_minus),

                    "central_diff_per_rho": float(central),
                    "forward_diff_per_rho": float(fwd),
                    "backward_diff_per_rho": float(bwd),
                    "disagreement_ratio": float(disagreement) if np.isfinite(disagreement) else np.nan,

                    "n_clip_plus": n_clip_plus,
                    "n_clip_minus": n_clip_minus,

                    # small direction diagnostics
                    "dir_norm": float(np.linalg.norm(d)),
                    "dir_seeded": True,
                    "rng_seed": int(rng_seed),
                }

                all_rows.append(row)
                completed_keys.add((start_name, float(eps_GHz), int(d_idx)))

                print(
                    f"    dir {d_idx+1:02d}/{n_dirs}: "
                    f"f0a={f_center_1:+.4f}, f+={f_plus:+.4f}, f-={f_minus:+.4f}, f0b={f_center_2:+.4f}, "
                    f"f0avg={f_center:+.4f}, "
                    f"disagree={disagreement if np.isfinite(disagreement) else np.nan:.3f}, "
                    f"clips(+/-)=({n_clip_plus},{n_clip_minus})"
                )

                # Checkpoint save after every successful direction
                save_checkpoint(save_dir, run_stamp, all_rows, summary_rows, D, tag="checkpoint")

            except Exception as e:
                print(
                    f"\n[ERROR] start={start_name}, eps_GHz={eps_GHz}, dir_idx={d_idx} failed: {e}"
                )
                # Save whatever progress exists before stopping
                all_ckpt_csv, summary_ckpt_csv, dirs_ckpt_csv = save_checkpoint(
                    save_dir, run_stamp, all_rows, summary_rows, D, tag="PARTIAL"
                )
                print("[partial saved]")
                print(" ", all_ckpt_csv)
                print(" ", summary_ckpt_csv)
                print(" ", dirs_ckpt_csv)
                raise

    # center at end (kept; still useful as a coarse drift indicator)
    f_center_end, _ = evaluate_wvg_db_retry(
        PMM, rho0, fpm, k, S, f_target_GHz, df_GHz, norms=norms, duty_cycle=duty_cycle
    )
    print(f"\n[{start_name}] center(end) = {f_center_end:.6f}")
    print(f"[{start_name}] center drift (begin->end) = {f_center_end - f_center_begin:+.6f}")

    # Rebuild summary for this start from all rows currently in memory
    summary_rows = [r for r in summary_rows if r.get("start_name") != start_name]

    df_start = pd.DataFrame([r for r in all_rows if r["start_name"] == start_name])
    if not df_start.empty:
        grp = df_start.groupby("eps_GHz", dropna=False)["disagreement_ratio"]

        for eps_val, s in grp:
            summary_rows.append({
                "run_stamp": run_stamp,
                "start_name": start_name,
                "eps_GHz": float(eps_val),
                "n_dirs": int(s.shape[0]),
                "disagreement_mean": float(np.nanmean(s.values)),
                "disagreement_median": float(np.nanmedian(s.values)),
                "disagreement_std": float(np.nanstd(s.values)),
                "disagreement_max": float(np.nanmax(s.values)),
                "disagreement_min": float(np.nanmin(s.values)),
                "f_center_begin": float(f_center_begin),
                "f_center_end": float(f_center_end),
                "center_drift": float(f_center_end - f_center_begin),
            })

    summary_rows = dedupe_summary_rows(summary_rows)

    # Checkpoint after completed start (includes summary rows)
    save_checkpoint(save_dir, run_stamp, all_rows, summary_rows, D, tag="checkpoint")
    print(f"[checkpoint summary saved after start={start_name}]")

# ----------------------------
# Save outputs (final)
# ----------------------------
df_all = pd.DataFrame(all_rows)
df_summary = pd.DataFrame(summary_rows)

all_csv = os.path.join(save_dir, f"epsilon_calibration_all_{run_stamp}.csv")
summary_csv = os.path.join(save_dir, f"epsilon_calibration_summary_{run_stamp}.csv")
dirs_csv = os.path.join(save_dir, f"epsilon_calibration_directions_{run_stamp}.csv")

df_all.to_csv(all_csv, index=False)
df_summary.to_csv(summary_csv, index=False)
pd.DataFrame(D).to_csv(dirs_csv, index=False)

print("\nSaved:")
print(" ", all_csv)
print(" ", summary_csv)
print(" ", dirs_csv)

# quick view
if not df_summary.empty:
    display(df_summary.sort_values(["start_name", "eps_GHz"]))
else:
    print("[info] Summary is empty (run may have stopped before any start summary completed).")

In [ ]:
# Finding Epsilon -- 6-port version for Naman active-subspace experiment

#MAKE SURE TO EMPTY OUTPUT FOLDER FIRST

# This is for collecting preliminary epsilon-calibration data
# 6-port version:
#   Trc1 = S21
#   Trc2 = S31
#   Trc3 = S41
#   Trc4 = S51
#   Trc5 = S61
#
# Active-subspace objective: integrated target-port power over the target frequency band
#
# For each task, change:
#   target_sparam = "S21" or "S31" or "S41" or "S51" or "S61"
#
# IMPORTANT:
#   Use resume_run_stamp = None for a fresh 6-port run.
#   Do not resume old 3-port epsilon checkpoints.

import os
import sys
import numpy as np
import pandas as pd
from datetime import datetime
import time

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from PMM.PMMInSitu import PMMInSitu

try:
    from RsInstrument import RsInstrument
except Exception:
    RsInstrument = None

# ----------------------------
# User settings
# ----------------------------
conf_file = '../confs/conf_test.yaml'
conf_dir = '../confs/'

save_dir = '../outputs/bayes_exploration'
os.makedirs(save_dir, exist_ok=True)

# ----------------------------
# 6-port objective settings
# ----------------------------
f_target_GHz = 6.0
df_GHz = 0.5
duty_cycle = 0.5

# Assumed VNA trace order
rx_labels = ["S21", "S31", "S41", "S51", "S61"]

# Change this for each beam-steering task
# Example:
#   "S21" = straight-through / port 2 target
#   "S31" = port 3 target
#   "S41" = port 4 target
#   "S51" = port 5 target
#   "S61" = port 6 target
target_sparam = "S21"

# Naman active-subspace recommended scalar objective:
# integrated target-port power
objective_name = "target_power_6port"

# Optional alternative, closer to your Bayes steering objective:
# objective_name = "target_minus_leakage_6port"

# ----------------------------
# Hardware mapping settings
# ----------------------------
fpm = 14.65  # GHz max plasma frequency parameter
k = 0.33
S = 1.0

# Epsilon candidates specified in GHz-equivalent scale, then converted to rho-space
eps_GHz_list = np.array([0.3, 0.1, 0.03], dtype=float)

# Direction sampling
n_dirs = 8
rng_seed = 20260224
rng = np.random.default_rng(rng_seed)

# Resume settings
# Use None for a fresh 6-port epsilon run.
resume_run_stamp = None

starts_to_run = [ ############################################# comment out the ones i dont need... 
    "optimized",
    "intuitive",
    "random",
    "cold",
]

resume_skip_completed = True
skip_start_if_summary_exists = False

# ----------------------------
# Initialize PMM
# ----------------------------
PMM = PMMInSitu(conf_file, conf_dir=conf_dir)

num_bulbs = sum(len(v) for v in PMM.config["serial_ports"].values())
rho_upper = PMM.f_a(fpm)

print(f"num_bulbs = {num_bulbs}")
print(f"rho_upper = PMM.f_a({fpm} GHz) = {rho_upper:.6f} rho-units")
print(f"target_sparam = {target_sparam}")
print(f"objective_name = {objective_name}")

# Convert epsilon list from GHz to rho-space
eps_rho_list = np.array([PMM.f_a(eps) for eps in eps_GHz_list], dtype=float)

print("Epsilon candidates:")
for eG, er in zip(eps_GHz_list, eps_rho_list):
    print(f"  {eG:.3f} GHz  ->  {er:.6e} rho-units")

# Interior bounds based on largest epsilon
eps_rho_max = float(np.max(eps_rho_list))
rho_lo_interior = eps_rho_max
rho_hi_interior = rho_upper - eps_rho_max

if rho_hi_interior <= rho_lo_interior:
    raise ValueError(
        "Interior bounds are invalid. Your largest epsilon is too large for rho_upper. "
        "Reduce eps_GHz_list or increase fpm."
    )

print(f"Interior bounds: [{rho_lo_interior:.6e}, {rho_hi_interior:.6e}]")

# ----------------------------
# Helper functions
# ----------------------------
def unit_random_directions(n, dim, rng):
    D = rng.standard_normal((n, dim))
    norms = np.linalg.norm(D, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return D / norms


def clip_rho(rho, lo=0.0, hi=None):
    if hi is None:
        return np.clip(rho, lo, None)
    return np.clip(rho, lo, hi)


def make_interior(rho, lo, hi):
    return np.clip(rho, lo, hi)


def get_6port_traces(PMM):
    """
    Read the 5 receive-port traces for the 6-port experiment.

    Preferred:
        use PMM.Get_S21_S31_S41_S51_S61() if you added that method.

    Fallback:
        directly query Trc1-Trc5 from the VNA.
    """
    # Preferred method if it exists in PMMInSitu.py
    if hasattr(PMM, "Get_S21_S31_S41_S51_S61"):
        freq, S21, S31, S41, S51, S61 = PMM.Get_S21_S31_S41_S51_S61()
        traces = {
            "S21": S21,
            "S31": S31,
            "S41": S41,
            "S51": S51,
            "S61": S61,
        }
        return freq, traces

    # Fallback direct VNA query
    if RsInstrument is None:
        raise RuntimeError(
            "RsInstrument is not available, and PMM.Get_S21_S31_S41_S51_S61() "
            "does not exist. Add the PMM method or fix RsInstrument import."
        )

    max_attempts = 6

    for attempt in range(max_attempts):
        try:
            instr = RsInstrument(PMM.VNA)

            instr.write_str('TRIGger1:SEQuence:SOURce IMM')
            time.sleep(7)

            S21_str = instr.query_str('CALC1:DATA:TRAC? "Trc1", FDAT')
            S31_str = instr.query_str('CALC1:DATA:TRAC? "Trc2", FDAT')
            S41_str = instr.query_str('CALC1:DATA:TRAC? "Trc3", FDAT')
            S51_str = instr.query_str('CALC1:DATA:TRAC? "Trc4", FDAT')
            S61_str = instr.query_str('CALC1:DATA:TRAC? "Trc5", FDAT')
            freq_str = instr.query_str('CALC1:DATA:STIM?')

            instr.write_str('TRIGger1:SEQuence:SOURce MAN')
            instr.close()

            if S21_str and S31_str and S41_str and S51_str and S61_str and freq_str:
                freq = np.array(freq_str.split(','), dtype=float)

                traces = {
                    "S21": np.array(S21_str.split(','), dtype=float),
                    "S31": np.array(S31_str.split(','), dtype=float),
                    "S41": np.array(S41_str.split(','), dtype=float),
                    "S51": np.array(S51_str.split(','), dtype=float),
                    "S61": np.array(S61_str.split(','), dtype=float),
                }

                return freq, traces

            else:
                raise ValueError("VNA returned empty data for one or more traces.")

        except Exception as e:
            print(f"Warning: VNA communication failed on attempt {attempt + 1}/{max_attempts}. Error: {e}")
            if attempt < max_attempts - 1:
                time.sleep(2)
            else:
                raise RuntimeError(
                    "VNA communication failed after multiple attempts. "
                    "Check VNA connection, trace setup, and VISA settings."
                )


def compute_6port_objective(freq_Hz, traces, target_sparam,
                            f_target_GHz, df_GHz,
                            objective_name="target_power_6port"):
    """
    Convert 6-port VNA traces into one scalar objective value.

    VNA traces are assumed to be in dB.
    Since S_dB = 20 log10(|S|), linear power is:
        |S|^2 = 10^(S_dB/10)

    target_power_6port:
        objective = integrated target-port power over frequency band

    target_minus_leakage_6port:
        objective = integrated target power minus integrated bad-port leakage
    """
    if target_sparam not in rx_labels:
        raise ValueError(f"target_sparam must be one of {rx_labels}, got {target_sparam}")

    freq_GHz = freq_Hz / 1e9

    f_lo = f_target_GHz - df_GHz / 2
    f_hi = f_target_GHz + df_GHz / 2

    band_mask = (freq_GHz >= f_lo) & (freq_GHz <= f_hi)

    if not np.any(band_mask):
        raise ValueError(
            f"No VNA frequency points found in band [{f_lo}, {f_hi}] GHz. "
            "Check f_target_GHz, df_GHz, and VNA sweep range."
        )

    freq_band_Hz = freq_Hz[band_mask]

    S_target_dB = traces[target_sparam][band_mask]
    P_target = 10.0 ** (S_target_dB / 10.0)

    if objective_name == "target_power_6port":
        # Integrated power. If only one point, fall back to sum.
        if len(freq_band_Hz) > 1:
            obj = np.trapz(P_target, freq_band_Hz)
        else:
            obj = np.sum(P_target)
        return float(obj)

    elif objective_name == "target_minus_leakage_6port":
        bad_sparams = [s for s in rx_labels if s != target_sparam]

        P_bad_total = np.zeros_like(P_target)
        for bad in bad_sparams:
            S_bad_dB = traces[bad][band_mask]
            P_bad_total += 10.0 ** (S_bad_dB / 10.0)

        score = P_target - P_bad_total

        if len(freq_band_Hz) > 1:
            obj = np.trapz(score, freq_band_Hz)
        else:
            obj = np.sum(score)

        return float(obj)

    else:
        raise ValueError(f"Unknown objective_name: {objective_name}")


def evaluate_wvg_6port(PMM, rho, fpm, k, S,
                       f_GHz, df_GHz,
                       norms=None,
                       duty_cycle=0.5):
    """
    One scalar objective evaluation for the 6-port epsilon experiment.

    Returns:
        obj, norms

    norms are kept only so the rest of the old epsilon code needs minimal edits.
    For target_power_6port, norms are not used.
    """
    PMM.ArraySet_Rho(rho, PMM.f_a(fpm), knob=k, scale=S)
    time.sleep(1)

    freq_Hz, traces = get_6port_traces(PMM)

    PMM.Deactivate_Bulb("all")
    time.sleep(1)
    PMM.Deactivate_Bulb("all")

    # Preserve the same duty-cycle wait style as your old epsilon cell.
    time.sleep(18 / duty_cycle - 20)

    obj = compute_6port_objective(
        freq_Hz=freq_Hz,
        traces=traces,
        target_sparam=target_sparam,
        f_target_GHz=f_GHz,
        df_GHz=df_GHz,
        objective_name=objective_name,
    )

    return obj, norms


def evaluate_wvg_6port_retry(PMM, rho, fpm, k, S,
                             f_GHz, df_GHz,
                             norms=None,
                             duty_cycle=0.5,
                             max_tries=10,
                             sleep_s=1.0):
    """
    Same as evaluate_wvg_6port, but retries if we hit:
        "device not configured"
    """
    last_err = None

    for attempt in range(1, max_tries + 1):
        try:
            return evaluate_wvg_6port(
                PMM, rho, fpm, k, S,
                f_GHz, df_GHz,
                norms=norms,
                duty_cycle=duty_cycle,
            )

        except Exception as e:
            last_err = e
            msg = str(e).lower()

            if "device not configured" in msg:
                print(f"[retry] device not configured (attempt {attempt}/{max_tries})")
                if attempt < max_tries:
                    time.sleep(sleep_s)
                    continue
                raise RuntimeError(f"device not configured after {max_tries} tries") from e

            else:
                raise

    raise last_err


def central_fwd_bwd_disagreement(f_plus, f_minus, f_center_avg, eps):
    """
    Returns central, forward, backward finite differences and disagreement.

    disagreement = |fwd - bwd| / |central|
    """
    central = (f_plus - f_minus) / (2.0 * eps)
    fwd = (f_plus - f_center_avg) / eps
    bwd = (f_center_avg - f_minus) / eps

    if abs(central) > 1e-12:
        disagreement = abs(fwd - bwd) / abs(central)
    else:
        disagreement = np.nan

    return central, fwd, bwd, disagreement


def save_checkpoint(save_dir, run_stamp, all_rows, summary_rows, D, tag="checkpoint"):
    """
    Save partial progress so hardware errors don't lose the whole run.
    """
    df_all_ckpt = pd.DataFrame(all_rows)
    df_summary_ckpt = pd.DataFrame(summary_rows)

    all_ckpt_csv = os.path.join(save_dir, f"epsilon_calibration_all_{run_stamp}_{tag}.csv")
    summary_ckpt_csv = os.path.join(save_dir, f"epsilon_calibration_summary_{run_stamp}_{tag}.csv")
    dirs_ckpt_csv = os.path.join(save_dir, f"epsilon_calibration_directions_{run_stamp}_{tag}.csv")

    df_all_ckpt.to_csv(all_ckpt_csv, index=False)
    df_summary_ckpt.to_csv(summary_ckpt_csv, index=False)
    pd.DataFrame(D).to_csv(dirs_ckpt_csv, index=False)

    return all_ckpt_csv, summary_ckpt_csv, dirs_ckpt_csv


def dedupe_summary_rows(summary_rows):
    """
    Keep only the latest summary entry for each (start_name, eps_GHz).
    """
    if len(summary_rows) == 0:
        return summary_rows

    df = pd.DataFrame(summary_rows)

    if df.empty:
        return summary_rows

    df = df.drop_duplicates(subset=["start_name", "eps_GHz"], keep="last")
    return df.to_dict(orient="records")


# ----------------------------
# Define starts
# ----------------------------

# 1) cold start
rho_cold = np.zeros(num_bulbs, dtype=float)

# 2) intuitive start
rho_intuitive = np.zeros(num_bulbs, dtype=float)

bulbs_to_turn_on = [
    41, 52, 62, 71, 79, 86,
    1, 2, 3, 4, 5, 6, 9, 10, 11, 12, 13,
    18, 19, 20, 21, 27, 28, 29, 30,
    37, 38, 39, 40, 48, 49, 50, 51,
    59, 60, 61, 68, 69, 70,
    77, 78, 84, 85, 91
]

fp_intuitive_on = 0.75 * fpm
rho_intuitive_on = PMM.f_a(fp_intuitive_on)

idx_on = np.array(bulbs_to_turn_on, dtype=int) - 1

if np.any(idx_on < 0) or np.any(idx_on >= num_bulbs):
    bad = [b for b in bulbs_to_turn_on if (b < 1 or b > num_bulbs)]
    raise ValueError(f"Invalid bulb addresses in bulbs_to_turn_on: {bad}")

rho_intuitive[idx_on] = rho_intuitive_on

print(f"Intuitive start: {len(idx_on)} bulbs ON at fp={fp_intuitive_on:.3f} GHz "
      f"(rho={rho_intuitive_on:.6e}), others OFF")

# 3) optimized start
optimized_csv_name = 'rho_best_manual_6GHz.csv'
optimized_csv_path = os.path.join(save_dir, optimized_csv_name)

optimized_row = 27

if not os.path.isfile(optimized_csv_path):
    raise FileNotFoundError(
        f"Could not find optimized start CSV: {optimized_csv_path}\n"
        "Put the file in save_dir and update optimized_csv_name."
    )

rho_csv = np.loadtxt(optimized_csv_path, delimiter=',')
rho_csv = np.array(rho_csv, dtype=float)

if rho_csv.ndim == 1:
    rho_optimized = rho_csv.copy()
    print("[info] optimized CSV is 1D (single row). Using that row.")

elif rho_csv.ndim == 2:
    n_rows, n_cols = rho_csv.shape

    if not (-n_rows <= optimized_row < n_rows):
        raise IndexError(f"optimized_row={optimized_row} out of range for CSV with {n_rows} rows")

    rho_optimized = rho_csv[optimized_row, :].copy()
    print(f"[info] optimized CSV shape={rho_csv.shape}; using row {optimized_row}.")

else:
    raise ValueError(f"Unexpected optimized CSV shape: {rho_csv.shape}")

if rho_optimized.size != num_bulbs:
    raise ValueError(
        f"optimized rho length {rho_optimized.size} does not match num_bulbs {num_bulbs}"
    )

rho_optimized = np.clip(rho_optimized, 0.0, rho_upper)

# 4) random start
rho_random = rng.uniform(low=0.0, high=rho_upper, size=num_bulbs)

# Clip all starts to valid range
rho_cold = clip_rho(rho_cold, 0.0, rho_upper)
rho_intuitive = clip_rho(rho_intuitive, 0.0, rho_upper)
rho_optimized = clip_rho(rho_optimized, 0.0, rho_upper)
rho_random = clip_rho(rho_random, 0.0, rho_upper)

# Push starts into interior box so rho0 +/- eps has room.
# Note: this means "cold" becomes small-but-nonzero instead of exactly zero.
rho_cold = make_interior(rho_cold, rho_lo_interior, rho_hi_interior)
rho_intuitive = make_interior(rho_intuitive, rho_lo_interior, rho_hi_interior)
rho_optimized = make_interior(rho_optimized, rho_lo_interior, rho_hi_interior)
rho_random = make_interior(rho_random, rho_lo_interior, rho_hi_interior)

starts_all = {
    "optimized": rho_optimized,
    "intuitive": rho_intuitive,
    "cold": rho_cold,
    "random": rho_random,
}

starts = {}

for nm in starts_to_run:
    if nm not in starts_all:
        raise ValueError(f"Unknown start name in starts_to_run: {nm}")
    starts[nm] = starts_all[nm]

# ----------------------------
# Main epsilon-finding loop
# ----------------------------
all_rows = []
summary_rows = []

# Same directions across starts
D = unit_random_directions(n_dirs, num_bulbs, rng)

# Run stamp handling
if resume_run_stamp is None:
    run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
else:
    run_stamp = str(resume_run_stamp)

print(f"\nRun stamp: {run_stamp}")

completed_keys = set()
summary_done_starts = set()

ckpt_all_path = os.path.join(save_dir, f"epsilon_calibration_all_{run_stamp}_checkpoint.csv")
ckpt_summary_path = os.path.join(save_dir, f"epsilon_calibration_summary_{run_stamp}_checkpoint.csv")
ckpt_dirs_path = os.path.join(save_dir, f"epsilon_calibration_directions_{run_stamp}_checkpoint.csv")

if resume_run_stamp is not None:
    if os.path.isfile(ckpt_all_path):
        df_prev_all = pd.read_csv(ckpt_all_path)
        print(f"[resume] Loaded checkpoint raw rows: {len(df_prev_all)} from")
        print("  ", ckpt_all_path)

        all_rows = df_prev_all.to_dict(orient="records")

        for _, rr in df_prev_all.iterrows():
            try:
                key = (str(rr["start_name"]), float(rr["eps_GHz"]), int(rr["dir_idx"]))
                completed_keys.add(key)
            except Exception:
                pass
    else:
        print(f"[resume] No raw checkpoint found at {ckpt_all_path}. Starting from scratch under this run_stamp.")

    if os.path.isfile(ckpt_summary_path) and os.path.getsize(ckpt_summary_path) > 0:
        try:
            df_prev_summary = pd.read_csv(ckpt_summary_path)
            print(f"[resume] Loaded checkpoint summary rows: {len(df_prev_summary)} from")
            print("  ", ckpt_summary_path)

            summary_rows = df_prev_summary.to_dict(orient="records")

            if not df_prev_summary.empty and "start_name" in df_prev_summary.columns:
                summary_done_starts = set(df_prev_summary["start_name"].astype(str).unique().tolist())

        except pd.errors.EmptyDataError:
            print(f"[resume] Summary checkpoint is non-empty bytes but has no columns: {ckpt_summary_path}")
    else:
        print(f"[resume] No usable summary checkpoint at {ckpt_summary_path} (missing or empty).")

    if not os.path.isfile(ckpt_dirs_path):
        print(f"[resume] Warning: directions checkpoint not found at {ckpt_dirs_path}. "
              "We will regenerate D from the fixed RNG seed.")

# Main loop
for start_name, rho0 in starts.items():

    if skip_start_if_summary_exists and (start_name in summary_done_starts):
        print("\n" + "="*90)
        print(f"START: {start_name} [SKIP summary already exists in checkpoint]")
        print("="*90)
        continue

    print("\n" + "="*90)
    print(f"START: {start_name}")
    print("="*90)

    # Center objective at beginning
    f_center_begin, norms = evaluate_wvg_6port_retry(
        PMM, rho0, fpm, k, S,
        f_target_GHz, df_GHz,
        norms=None,
        duty_cycle=duty_cycle
    )

    print(f"[{start_name}] center(begin) = {f_center_begin:.6e}")

    for eps_GHz, eps_rho in zip(eps_GHz_list, eps_rho_list):

        print(f"\n  Epsilon = {eps_GHz:.3f} GHz-equivalent  (rho eps = {eps_rho:.6e})")

        for d_idx in range(n_dirs):
            d = D[d_idx]

            resume_key = (start_name, float(eps_GHz), int(d_idx))

            if resume_skip_completed and (resume_key in completed_keys):
                print(f"    dir {d_idx+1:02d}/{n_dirs}: [SKIP already saved]")
                continue

            try:
                raw_plus = rho0 + eps_rho * d
                raw_minus = rho0 - eps_rho * d

                rho_plus = clip_rho(raw_plus, 0.0, rho_upper)
                rho_minus = clip_rho(raw_minus, 0.0, rho_upper)

                n_clip_plus = int(np.sum((raw_plus < 0.0) | (raw_plus > rho_upper)))
                n_clip_minus = int(np.sum((raw_minus < 0.0) | (raw_minus > rho_upper)))

                # Center before perturbations
                f_center_1, _ = evaluate_wvg_6port_retry(
                    PMM, rho0, fpm, k, S,
                    f_target_GHz, df_GHz,
                    norms=norms,
                    duty_cycle=duty_cycle
                )

                # Plus
                f_plus, _ = evaluate_wvg_6port_retry(
                    PMM, rho_plus, fpm, k, S,
                    f_target_GHz, df_GHz,
                    norms=norms,
                    duty_cycle=duty_cycle
                )

                # Minus
                f_minus, _ = evaluate_wvg_6port_retry(
                    PMM, rho_minus, fpm, k, S,
                    f_target_GHz, df_GHz,
                    norms=norms,
                    duty_cycle=duty_cycle
                )

                # Center after perturbations
                f_center_2, _ = evaluate_wvg_6port_retry(
                    PMM, rho0, fpm, k, S,
                    f_target_GHz, df_GHz,
                    norms=norms,
                    duty_cycle=duty_cycle
                )

                f_center = 0.5 * (f_center_1 + f_center_2)

                central, fwd, bwd, disagreement = central_fwd_bwd_disagreement(
                    f_plus=f_plus,
                    f_minus=f_minus,
                    f_center_avg=f_center,
                    eps=eps_rho
                )

                signal = abs(f_plus - f_minus)

                row = {
                    "run_stamp": run_stamp,
                    "start_name": start_name,
                    "objective": objective_name,
                    "target_sparam": target_sparam,
                    "rx_labels": ",".join(rx_labels),

                    "f_target_GHz": f_target_GHz,
                    "df_GHz": df_GHz,
                    "fpm_GHz": fpm,
                    "k": k,
                    "S": S,

                    "dir_idx": d_idx,
                    "eps_GHz": float(eps_GHz),
                    "eps_rho": float(eps_rho),

                    "f_center_begin": float(f_center),
                    "f_center_1": float(f_center_1),
                    "f_center_2": float(f_center_2),
                    "f_plus": float(f_plus),
                    "f_minus": float(f_minus),
                    "signal_abs_fplus_minus_fminus": float(signal),

                    "central_diff_per_rho": float(central),
                    "forward_diff_per_rho": float(fwd),
                    "backward_diff_per_rho": float(bwd),
                    "disagreement_ratio": float(disagreement) if np.isfinite(disagreement) else np.nan,

                    "n_clip_plus": n_clip_plus,
                    "n_clip_minus": n_clip_minus,

                    "dir_norm": float(np.linalg.norm(d)),
                    "dir_seeded": True,
                    "rng_seed": int(rng_seed),
                }

                all_rows.append(row)
                completed_keys.add((start_name, float(eps_GHz), int(d_idx)))

                print(
                    f"    dir {d_idx+1:02d}/{n_dirs}: "
                    f"f0a={f_center_1:+.4e}, "
                    f"f+={f_plus:+.4e}, "
                    f"f-={f_minus:+.4e}, "
                    f"f0b={f_center_2:+.4e}, "
                    f"f0avg={f_center:+.4e}, "
                    f"signal={signal:.4e}, "
                    f"disagree={disagreement if np.isfinite(disagreement) else np.nan:.3f}, "
                    f"clips(+/-)=({n_clip_plus},{n_clip_minus})"
                )

                save_checkpoint(save_dir, run_stamp, all_rows, summary_rows, D, tag="checkpoint")

            except Exception as e:
                print(
                    f"\n[ERROR] start={start_name}, eps_GHz={eps_GHz}, dir_idx={d_idx} failed: {e}"
                )

                all_ckpt_csv, summary_ckpt_csv, dirs_ckpt_csv = save_checkpoint(
                    save_dir, run_stamp, all_rows, summary_rows, D, tag="PARTIAL"
                )

                print("[partial saved]")
                print(" ", all_ckpt_csv)
                print(" ", summary_ckpt_csv)
                print(" ", dirs_ckpt_csv)

                raise

    # Center at end
    f_center_end, _ = evaluate_wvg_6port_retry(
        PMM, rho0, fpm, k, S,
        f_target_GHz, df_GHz,
        norms=norms,
        duty_cycle=duty_cycle
    )

    print(f"\n[{start_name}] center(end) = {f_center_end:.6e}")
    print(f"[{start_name}] center drift (begin->end) = {f_center_end - f_center_begin:+.6e}")

    # Rebuild summary for this start
    summary_rows = [r for r in summary_rows if r.get("start_name") != start_name]

    df_start = pd.DataFrame([r for r in all_rows if r["start_name"] == start_name])

    if not df_start.empty:
        grp = df_start.groupby("eps_GHz", dropna=False)

        for eps_val, gdf in grp:
            s = gdf["disagreement_ratio"]

            summary_rows.append({
                "run_stamp": run_stamp,
                "start_name": start_name,
                "objective": objective_name,
                "target_sparam": target_sparam,

                "eps_GHz": float(eps_val),
                "n_dirs": int(s.shape[0]),

                "disagreement_mean": float(np.nanmean(s.values)),
                "disagreement_median": float(np.nanmedian(s.values)),
                "disagreement_std": float(np.nanstd(s.values)),
                "disagreement_max": float(np.nanmax(s.values)),
                "disagreement_min": float(np.nanmin(s.values)),

                "signal_mean": float(np.nanmean(gdf["signal_abs_fplus_minus_fminus"].values)),
                "signal_median": float(np.nanmedian(gdf["signal_abs_fplus_minus_fminus"].values)),
                "signal_min": float(np.nanmin(gdf["signal_abs_fplus_minus_fminus"].values)),
                "signal_max": float(np.nanmax(gdf["signal_abs_fplus_minus_fminus"].values)),

                "f_center_begin": float(f_center_begin),
                "f_center_end": float(f_center_end),
                "center_drift": float(f_center_end - f_center_begin),
            })

    summary_rows = dedupe_summary_rows(summary_rows)

    save_checkpoint(save_dir, run_stamp, all_rows, summary_rows, D, tag="checkpoint")
    print(f"[checkpoint summary saved after start={start_name}]")

# ----------------------------
# Save final outputs
# ----------------------------
df_all = pd.DataFrame(all_rows)
df_summary = pd.DataFrame(summary_rows)

all_csv = os.path.join(save_dir, f"epsilon_calibration_all_{run_stamp}.csv")
summary_csv = os.path.join(save_dir, f"epsilon_calibration_summary_{run_stamp}.csv")
dirs_csv = os.path.join(save_dir, f"epsilon_calibration_directions_{run_stamp}.csv")

df_all.to_csv(all_csv, index=False)
df_summary.to_csv(summary_csv, index=False)
pd.DataFrame(D).to_csv(dirs_csv, index=False)

print("\nSaved:")
print(" ", all_csv)
print(" ", summary_csv)
print(" ", dirs_csv)

if not df_summary.empty:
    display(df_summary.sort_values(["start_name", "eps_GHz"]))
else:
    print("[info] Summary is empty. Run may have stopped before any start summary completed.")

# f0a      = objective at starting point before plus/minus
# f+       = objective after positive perturbation
# f-       = objective after negative perturbation
# f0b      = objective at starting point after plus/minus
# f0avg    = average center objective
# signal   = |f+ - f-|, the size of the finite-difference signal
# disagree = how different forward/backward slopes are
# clips    = how many rho entries hit 0 or rho_upper after perturbation

num_bulbs = 91
rho_upper = PMM.f_a(14.65 GHz) = 1.050645 rho-units
target_sparam = S21
objective_name = target_power_6port
Epsilon candidates:
  0.300 GHz  ->  2.151491e-02 rho-units
  0.100 GHz  ->  7.171638e-03 rho-units
  0.030 GHz  ->  2.151491e-03 rho-units
Interior bounds: [2.151491e-02, 1.029130e+00]
Intuitive start: 44 bulbs ON at fp=10.988 GHz (rho=7.879837e-01), others OFF
[info] optimized CSV shape=(28, 91); using row 27.

Run stamp: 20260616_115443

START: optimized


/var/folders/2s/phrk5g316q3dsdx1nd1zf7r00000gq/T/ipykernel_92831/3185774769.py:269: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  obj = np.trapz(P_target, freq_band_Hz)


[optimized] center(begin) = 4.391135e+04

  Epsilon = 0.300 GHz-equivalent  (rho eps = 2.151491e-02)
    dir 01/8: f0a=+5.8848e+04, f+=+7.3043e+04, f-=+8.0655e+04, f0b=+7.8514e+04, f0avg=+6.8681e+04, signal=7.6112e+03, disagree=4.293, clips(+/-)=(0,0)


/var/folders/2s/phrk5g316q3dsdx1nd1zf7r00000gq/T/ipykernel_92831/3185774769.py:269: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  obj = np.trapz(P_target, freq_band_Hz)


    dir 02/8: f0a=+7.3581e+04, f+=+7.4988e+04, f-=+8.0205e+04, f0b=+7.6912e+04, f0avg=+7.5246e+04, signal=5.2172e+03, disagree=1.802, clips(+/-)=(0,0)


...

/var/folders/2s/phrk5g316q3dsdx1nd1zf7r00000gq/T/ipykernel_92831/3185774769.py:269: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  obj = np.trapz(P_target, freq_band_Hz)


    dir 04/8: f0a=+7.4370e+04, f+=+6.9711e+04, f-=+7.2201e+04, f0b=+7.3372e+04, f0avg=+7.3871e+04, signal=2.4905e+03, disagree=4.681, clips(+/-)=(0,0)


/var/folders/2s/phrk5g316q3dsdx1nd1zf7r00000gq/T/ipykernel_92831/3185774769.py:269: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  obj = np.trapz(P_target, freq_band_Hz)


    dir 05/8: f0a=+7.1961e+04, f+=+7.0367e+04, f-=+7.0207e+04, f0b=+6.9840e+04, f0avg=+7.0900e+04, signal=1.5936e+02, disagree=15.395, clips(+/-)=(0,0)


/var/folders/2s/phrk5g316q3dsdx1nd1zf7r00000gq/T/ipykernel_92831/3185774769.py:269: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  obj = np.trapz(P_target, freq_band_Hz)


In [ ]:
# Unit Test: RS-485

import os
import sys
import time
import unittest
import numpy as np

# Make PMM import work like your scripts
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from PMM.PMMInSitu import PMMInSitu


class TestRS485BusHealth(unittest.TestCase):
    """
    Integration test: probes each RS485 bus by forcing a benign hardware transaction.
    Goal: identify which bus throws "device not configured" / IO errors.

    Run:
      python -m unittest -v test_rs485_bus_health.py
    """

    # ---- match your usual settings ----
    conf_file = "../confs/conf_test.yaml"
    conf_dir = "../confs/"

    # Keep these consistent with your normal calls (only used to trigger a measurement transaction)
    fpm = 14.65
    k = 0.33
    S = 1.0
    f_target_GHz = 6.0
    df_GHz = 0.5
    objective_name = "dB"
    duty_cycle = 0.5  # keep it as safe as possible; still triggers comms in most stacks

    # Probe behavior
    trials_per_bus = 10
    sleep_between_trials_s = 0.5

    @classmethod
    def setUpClass(cls):
        cls.PMM = PMMInSitu(cls.conf_file, conf_dir=cls.conf_dir)

        # Config format assumed from your code: dict of buses -> list of bulb addresses
        cls.serial_ports = cls.PMM.config.get("serial_ports", {})
        if not isinstance(cls.serial_ports, dict) or len(cls.serial_ports) == 0:
            raise RuntimeError("PMM.config['serial_ports'] missing/empty; cannot run RS485 bus health test.")

        # Total bulbs like your script
        cls.num_bulbs = sum(len(v) for v in cls.serial_ports.values())

        # Safe "all off" vector (still triggers writes/reads typically)
        cls.rho_all_off = np.zeros(cls.num_bulbs, dtype=float)

        # Precompute norms once (like your main script)
        # Two calls: first gets norms, second returns objective using those norms (same as your evaluate_wvg_db)
        _, cls.norms = cls.PMM.Wvg_Obj_Get(
            cls.rho_all_off,
            cls.fpm, cls.k, cls.S,
            cls.f_target_GHz, cls.df_GHz,
            objective=cls.objective_name,
            norms=[],
            duty_cycle=cls.duty_cycle,
        )

    def _probe_transaction(self):
        """
        One benign transaction that still exercises comms.
        """
        obj, _ = self.PMM.Wvg_Obj_Get(
            self.rho_all_off,
            self.fpm, self.k, self.S,
            self.f_target_GHz, self.df_GHz,
            objective=self.objective_name,
            norms=self.norms,
            duty_cycle=self.duty_cycle,
        )
        return obj

    def test_rs485_buses(self):
        """
        For each RS485 bus, try a few transactions while assuming PMM routes to that bus
        according to its internal mapping/config.

        If your PMM stack only talks to *all* buses every transaction, this still works:
        - the failing bus will still cause the global transaction to throw
        - we then isolate by temporarily probing *only* that bus (see NOTE below)
        """
        results = []  # list of dicts: bus, ok_count, fail_count, last_error

        # --- PASS 1: global probe repeated, but attributed per bus ---
        # This detects unreliability, but may not isolate perfectly if every call uses all buses.
        # We'll do a second isolation pass below if anything fails.
        for bus_name in self.serial_ports.keys():
            ok = 0
            fail = 0
            last_err = None

            for t in range(self.trials_per_bus):
                try:
                    _ = self._probe_transaction()
                    ok += 1
                except Exception as e:
                    fail += 1
                    last_err = repr(e)

                time.sleep(self.sleep_between_trials_s)

            results.append(
                {"bus": str(bus_name), "ok": ok, "fail": fail, "last_error": last_err}
            )

        # If nothing failed, we’re done.
        any_fail = any(r["fail"] > 0 for r in results)
        if not any_fail:
            # Print a compact report in verbose mode
            for r in results:
                print(f"[RS485] {r['bus']}: ok={r['ok']} fail={r['fail']}")
            return

        # --- PASS 2 (isolation): try to only exercise bulbs on one bus at a time ---
        # This requires that setting rho for bulbs on other buses to NaN (or leaving unchanged)
        # does NOT cause PMM to write to them. If your PMM ignores NaNs, this isolates well.
        #
        # If your PMM does not support NaNs, set everything to 0 and only toggle bulbs on that bus
        # by a tiny amount; sometimes the controller will only write changed channels.
        isolated_results = []

        for bus_name, addr_list in self.serial_ports.items():
            ok = 0
            fail = 0
            last_err = None

            # Attempt NaN-isolation first
            rho = np.full(self.num_bulbs, np.nan, dtype=float)
            # Turn “on” only this bus’s bulbs at 0 (still benign), but present as numbers
            idx = np.array(addr_list, dtype=int) - 1
            rho[idx] = 0.0

            for t in range(self.trials_per_bus):
                try:
                    # If PMM can’t handle NaNs, this may raise; we catch and fall back.
                    obj, _ = self.PMM.Wvg_Obj_Get(
                        rho,
                        self.fpm, self.k, self.S,
                        self.f_target_GHz, self.df_GHz,
                        objective=self.objective_name,
                        norms=self.norms,
                        duty_cycle=self.duty_cycle,
                    )
                    _ = obj
                    ok += 1
                except Exception as e:
                    # Fallback: try “tiny toggle” method
                    try:
                        rho2 = np.zeros(self.num_bulbs, dtype=float)
                        rho2[idx] = 1e-12  # tiny, should be effectively zero but may force a write on that bus
                        obj2, _ = self.PMM.Wvg_Obj_Get(
                            rho2,
                            self.fpm, self.k, self.S,
                            self.f_target_GHz, self.df_GHz,
                            objective=self.objective_name,
                            norms=self.norms,
                            duty_cycle=self.duty_cycle,
                        )
                        _ = obj2
                        ok += 1
                    except Exception as e2:
                        fail += 1
                        last_err = repr(e2)

                time.sleep(self.sleep_between_trials_s)

            isolated_results.append(
                {"bus": str(bus_name), "ok": ok, "fail": fail, "last_error": last_err, "n_bulbs": len(addr_list)}
            )

        # Print report
        print("\n=== RS485 BUS HEALTH REPORT ===")
        for r in isolated_results:
            status = "OK" if r["fail"] == 0 else "FAIL"
            print(f"{status:4s} | {r['bus']} | bulbs={r['n_bulbs']:3d} | ok={r['ok']} fail={r['fail']} | last={r['last_error']}")

        # Fail test if any bus fails at least once
        bad = [r for r in isolated_results if r["fail"] > 0]
        self.assertEqual(
            len(bad), 0,
            msg="One or more RS485 buses appear unreliable. See report above."
        )


if __name__ == "__main__":
    unittest.main(argv=['first-arg-is-ignored'], exit=False, verbosity=2)